# Grain-boundary cleavage study (EAM/Fe)

Constructs a small bcc-Fe bicrystal by stacking two oriented
slabs, adds vacuum, and scans candidate cleavage planes around
the bicrystal midplane via `cleave_gb_structure` + a static
engine. Reports cleavage energy (J/m^2) as a function of plane
coordinate.


In [1]:
import pathlib
import numpy as np
from ase.build import bulk, stack
from ase.calculators.eam import EAM
from ase.lattice.cubic import BodyCenteredCubic as bcc

from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.engine import (
    ASEEngine,
    CalcInputMinimize,
    CalcInputStatic,
    calculate,
)
from pyiron_workflow_atomistics.physics.bulk import optimise_cubic_lattice_parameter
from pyiron_workflow_atomistics.physics.grain_boundary import cleave_gb_structure
from pyiron_workflow_atomistics.structure import add_vacuum

_pot_path = str(next(p for p in [
    pathlib.Path("Al-Fe.eam.fs"),
    pathlib.Path("notebooks/Al-Fe.eam.fs"),
    pathlib.Path(__file__).parent / "Al-Fe.eam.fs" if "__file__" in globals() else pathlib.Path("."),
] if p.exists()))

# Static engine for the cleavage scan (single-point energies).
engine_static = ASEEngine(
    EngineInput=CalcInputStatic(),
    calculator=EAM(potential=_pot_path),
    working_directory="./_gb_cleave_runs",
)

# Minimising engine reusing the same EAM potential, for bulk lat-opt.
engine_min = ASEEngine(
    EngineInput=CalcInputMinimize(
        force_convergence_tolerance=0.05, max_iterations=100
    ),
    calculator=EAM(potential=_pot_path),
    working_directory="./_gb_cleave_runs",
)


In [2]:
# Optimise the bcc Fe lattice parameter via Workflow so the bicrystal slab
# uses a self-consistent a0, rather than a hard-coded 2.85 A.
wf_setup = Workflow("gb_cleave_setup")
wf_setup.lat_opt = optimise_cubic_lattice_parameter(
    structure=bulk("Fe", a=2.85, cubic=True),
    name="Fe",
    crystalstructure="bcc",
    engine=engine_min.with_working_directory("lat_opt"),
    strain_range=(-0.02, 0.02),
    num_points=5,
)
wf_setup.run()

lc = float(wf_setup.lat_opt.outputs.a0.value)
print(f"optimised a0 = {lc:.4f} A (was hard-coded as 2.85 A)")

# Build a small S3-RA110 like bicrystal from oriented bcc slabs using the optimised lc.
surface1 = [1, 1, 1]
surface2 = [1, 1, -1]
rotation_axis = [1, -1, 0]
v1 = list(-np.cross(rotation_axis, surface1))
v2 = list(-np.cross(rotation_axis, surface2))

slab1 = bcc(symbol="Fe", latticeconstant=lc,
            directions=[rotation_axis, v1, surface1], size=[1, 1, 2])
slab2 = bcc(symbol="Fe", latticeconstant=lc,
            directions=[rotation_axis, v2, surface2], size=[1, 1, 2])
gb = stack(slab1, slab2)
gb_with_vacuum = add_vacuum.node_function(gb, vacuum_length=15.0, axis="c")
gb_z = gb_with_vacuum.cell[-1, -1]
area = float(np.linalg.norm(np.cross(gb_with_vacuum.cell[0], gb_with_vacuum.cell[1])))
print(f"bicrystal: {len(gb_with_vacuum)} atoms, c = {gb_z:.2f} A, area = {area:.2f} A^2")


optimised a0 = 2.8554 A (was hard-coded as 2.85 A)
bicrystal: 24 atoms, c = 24.07 A, area = 28.24 A^2


In [3]:
# Relax the uncleaved bicrystal so E_uncleaved is the equilibrium GB energy,
# wired through a small Workflow for parity with the lat-opt step.
# Without this, the as-stacked interface is high-strain and
# (E_cleaved - E_uncleaved) can go negative near the GB plane.
wf_ref = Workflow("gb_cleave_ref")
wf_ref.uncleaved = calculate(
    structure=gb_with_vacuum,
    engine=engine_min.with_working_directory("uncleaved"),
    label="uncleaved",
)
wf_ref.run()
gb_relaxed = wf_ref.uncleaved.outputs.engine_output.value.final_structure
E_uncleaved = wf_ref.uncleaved.outputs.engine_output.value.final_energy
print(f"uncleaved (relaxed) energy: {E_uncleaved:.3f} eV")


uncleaved (relaxed) energy: -87.160 eV


In [4]:
# Identify viable cleavage planes near the bicrystal midplane (on the relaxed structure).
gb_midplane = gb_z / 2.0
cleaved_structures, cleavage_coords = cleave_gb_structure.node_function(
    base_structure=gb_relaxed,
    axis_to_cleave="c",
    target_coord=gb_midplane,
    tol=0.3,
    cleave_region_halflength=1.5,
    layer_tolerance=0.3,
    separation=8.0,
    use_fractional=False,
)
print(f"found {len(cleaved_structures)} candidate cleavage plane(s)")
for c in cleavage_coords:
    print(f"  plane @ c = {c:.3f} A")


found 4 candidate cleavage plane(s)
  plane @ c = 11.018 A
  plane @ c = 11.873 A
  plane @ c = 13.033 A
  plane @ c = 13.876 A


In [5]:
# Evaluate each cleaved structure with the static engine, in its own subdir.
rows = []
for i, (struct, coord) in enumerate(zip(cleaved_structures, cleavage_coords)):
    sub_engine = engine_static.with_working_directory(f"cleavage/plane_{i:02d}")
    cell = calculate(structure=struct, engine=sub_engine)
    cell.run()
    e_cleaved = cell.outputs.engine_output.value.final_energy
    # Cleavage energy in J/m^2, dividing by two surfaces.
    cleavage_energy_eV = (e_cleaved - E_uncleaved) / (2 * area)
    cleavage_energy_Jm2 = cleavage_energy_eV * 16.021766208
    rows.append({"cleavage_coord": float(coord), "energy_eV": float(e_cleaved),
                  "cleavage_energy_Jm2": float(cleavage_energy_Jm2)})

import pandas as pd
df = pd.DataFrame(rows)
print(df)
print(f"min cleavage energy: {df.cleavage_energy_Jm2.min():.3f} J/m^2")


   cleavage_coord  energy_eV  cleavage_energy_Jm2
0       11.017779 -80.284888             1.950071
1       11.873074 -82.023242             1.457006
2       13.033464 -81.951645             1.477314
3       13.876038 -80.264574             1.955833
min cleavage energy: 1.457 J/m^2
